# Market-Aware Business Forecasting System

This notebook demonstrates an end-to-end workflow:

- Load and validate the dataset
- Preprocess and engineer time-series features
- Train a market-aware linear regression model
- Evaluate performance (MAE, RMSE)
- Visualize trends and model fit
- Forecast the next 30 days of brand sales

> Dataset: `../data/raw_data.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_preprocessing import preprocess_data
from src.feature_engineering import build_features
from src.model import train_linear_regression
from src.evaluate import regression_metrics
from src.forecasting import forecast_next_days

PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw_data.csv'

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
# Step 2: Preprocessing

df_clean, df_missing = preprocess_data(df_raw)
print('Clean rows:', len(df_clean))
print('Rows with any missing values (captured for inspection):', len(df_missing))

df_clean.tail()

In [ ]:
# Step 3: Feature Engineering

df_features = build_features(df_clean)
print('Feature rows after dropping lag/rolling NA:', len(df_features))

df_features[['date','brand_sales','category_sales','category_growth','lag_7','lag_30','rolling_mean_7']].head()

In [ ]:

# Step 4: Model Building (Time-based split)


feature_cols = [
    'category_sales',
    'category_growth',
    'last_month_sales',
    'last_year_sales',
    'lag_7',
    'lag_30',
]

split, model_result = train_linear_regression(
    df_features=df_features,
    feature_cols=feature_cols,
    target_col='brand_sales',
    test_size=60,
)

metrics = regression_metrics(model_result.y_true, model_result.y_pred)
metrics

In [ ]:
# Step 6: Visualization (matplotlib only)

# 1) Actual vs Predicted (holdout)
test_dates = df_features.sort_values('date').iloc[-len(model_result.y_true):]['date']

plt.figure(figsize=(10,4))
plt.plot(test_dates, model_result.y_true, label='Actual', linewidth=2)
plt.plot(test_dates, model_result.y_pred, label='Predicted', linewidth=2)
plt.title('Actual vs Predicted Brand Sales (Holdout)')
plt.xlabel('Date')
plt.ylabel('Brand Sales')
plt.legend()
plt.tight_layout()
plt.show()

# 2) Category vs Brand trend
plt.figure(figsize=(10,4))
plt.plot(df_clean['date'], df_clean['category_sales'], label='Category Sales', linewidth=2)
plt.plot(df_clean['date'], df_clean['brand_sales'], label='Brand Sales', linewidth=2)
plt.title('Category vs Brand Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.show()

# 3) Market share trend
plt.figure(figsize=(10,4))
plt.plot(df_clean['date'], df_clean['market_share'], label='Market Share', linewidth=2)
plt.title('Market Share Trend (Brand / Category)')
plt.xlabel('Date')
plt.ylabel('Market Share')
plt.ylim(0, max(0.4, float(df_clean['market_share'].max())*1.1))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Forecasting (Next 30 Days)

forecast_res = forecast_next_days(
    df_features=df_features,
    model=model_result.model,
    feature_cols=feature_cols,
    n_days=30,
)

forecast_res.forecast_df.head(10)

## Interpretation Tips (talk-track)

- **If brand sales rise but market share is flat**, the category is lifting all boats.
- **If market share rises**, your brand is outperforming the category (pricing, distribution, marketing effectiveness).
- The simple linear model is a strong baseline because it’s **explainable** and easy to communicate; you can extend it later with more drivers (price, promo, distribution, media spend).